# Calculating regression coefficients for individual conversations in a given dataset

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.linalg import lstsq
from scipy.stats import t as tdist

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
from datetime import datetime as dt
# import statsmodels.formula.api as smf
import warnings;warnings.filterwarnings('ignore')

In [2]:
OUTPUTS_NAME = 'results-{}-updated.csv'

In [3]:
# pre recent run
# DATA_PATH = '../data/ckpts/rosen'

DATA_PATH = '../../updated-data/obs/lme-ready'
NULL_DATA_PATH = '../../updated-data/null/null-lme-ready'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [5]:
fs = [f for f in os.listdir(DATA_PATH) if ('xling-' not in f) and (not f.startswith('.')) and f.endswith('.parquet')]
len(fs)

4518

## Processing individual datasets

In [6]:
Y_VAR = 'I'

In [7]:
param_list = [
    'nx',
    'ny'
]
processed_params = [
    'Intercept',
    'self',
    'other',
    # 'null',
    'turn_delta_self',
    'turn_delta_other'
]

In [8]:
df_scale, df_regression = [], []

#### Crunching numbers one individual speaker at a time

In [9]:
for f in tqdm(fs):
    # print(f.split('/')[-1])

    corpus = f.split('/')[-1].split('-')[0]
    cid = '-'.join(f.split('/')[-1].split('-')[1:])

    dfo = pq.ParquetFile(os.path.join(DATA_PATH, f))
    if f.startswith('grace-'):
        dfn = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f.replace('grace', 'Miao-fNIRS')))
    else:
        dfn = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f))

    df = [dfo.read(columns=[Y_VAR]+param_list+['x_speaker', 'y_speaker', 'x_turn_id', 'y_turn_id']).to_pandas()]
    df[-1]['null'] = False

    df += [dfn.read(columns=[Y_VAR]+param_list+['x_speaker', 'y_speaker', 'x_turn_id', 'y_turn_id']).to_pandas()]
    df[-1]['null'] = True
    df = pd.concat(df, ignore_index=True)

    # table = pq.ParquetFile(f)
    #
    # df = table.read(columns=[Y_VAR]+param_list+['x_speaker', 'y_speaker', 'x_turn_id', 'y_turn_id']).to_pandas()

    # dealing with temporal distance between turns
    df['turn_delta'] = df['y_turn_id'] - df['x_turn_id']
    df = df.loc[
        (df['turn_delta'] > 0)
        & (df['turn_delta'] <= 200)
    ]
    df['turn_delta'].loc[df['null']] = 0.
    df['turn_delta'] += 1
    df['turn_delta'] = np.log(df['turn_delta'])


    # setting up self and other cols
    df['self'] = (df['x_speaker'] == df['y_speaker'])
    df['other'] = ~df['self']

    # how we treat self and oter in null condition
    df['self'].loc[df['null']] = False
    df['other'].loc[df['null']] = False

    df['turn_delta_other'] = df['turn_delta']
    df['turn_delta_self'] = df['turn_delta']
    df['turn_delta_other'].loc[df['self']] = 0.
    df['turn_delta_self'].loc[df['other']] = 0.

    # convert T/F values to integers
    df['self'] = df['self'].astype(int)
    df['other'] = df['other'].astype(int)
    df['null'] = df['null'].astype(int)

    df['Intercept'] = 1.

    params, _, _, _ = lstsq(
        df[param_list + processed_params].values,
        df[Y_VAR].values
    )

    df_regression += [
        {
            'corpus': corpus,
            'cid': cid,
            'param': name,
            'beta': param,
            'k': len(df),
            'nspeakers': df['x_speaker'].nunique()
        }

    for name, param in list(zip(param_list + processed_params, params))]

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/4518 [00:00<?, ?it/s]

#### Last checks and saving outputs

In [ ]:
df_regression.head()

In [ ]:
# df_regression.isna().sum()

In [ ]:
# df_regression.to_csv(
#     os.path.join(OUTPUTS_PATH, str(OUTPUTS_NAME).format(dt.now().strftime("%Y%m%d%H%M"))),
#     index=False,
#     encoding='utf-8'
# )

## Aggregate Results

In [10]:
res = [
    {
        'param': param,
        'beta': df_regression['beta'].loc[df_regression['param'].isin([param])].mean(),
        'sd': df_regression['beta'].loc[df_regression['param'].isin([param])].std(),
        'k': df_regression['param'].isin([param]).sum()
    } for param in df_regression['param'].unique()
]

res = pd.DataFrame(res)

In [11]:
res['se'] = res['sd'] / np.sqrt(res['k'])
res['t'] = res['beta'] / res['se']
res['p'] = [1-tdist.cdf(t.__abs__(), dof-1).item() for t,dof in res[['t', 'k']].values]

In [12]:
res.head(n=10)

,param,beta,sd,k,se,t,p
0,nx,0.157351,0.084465,4518,0.001257,125.218020,0.000000e+00
1,ny,-0.027013,0.034500,4518,0.000513,-52.629710,0.000000e+00
2,Intercept,-0.044828,0.993197,4518,0.014776,-3.033803,1.214326e-03
3,self,-0.224157,1.375932,4518,0.020470,-10.950342,0.000000e+00
4,other,0.089724,1.029163,4518,0.015311,5.860013,2.478671e-09
5,turn_delta_self,0.020830,0.690688,4518,0.010276,2.027109,2.135493e-02
6,turn_delta_other,-0.013158,0.348804,4518,0.005189,-2.535517,5.630787e-03


#### Save results

In [13]:
res.to_csv(
    os.path.join(OUTPUTS_PATH, 'agg-params.csv'),
    index=False,
    encoding='utf-8'
)

## Corpus level analysis of results

In [14]:
split_by = ['corpus','param']

In [15]:
df0 = df_regression[split_by+['beta']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['beta']].groupby(by=split_by).agg('std').reset_index()['beta'].values

df0['k'] = df_regression[split_by+['cid']].groupby(by=split_by).agg('count').reset_index()['cid'].values
# df0['k'] = df_regression[split_by+['k']].groupby(by=split_by).agg('sum').reset_index()['k'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [16]:
df0['t'] = df0['beta'] / df0['se']
df0['p'] = [1-tdist.cdf(t.__abs__(), dof-1).item() for t,dof in df0[['t', 'k']].values]

In [17]:
df0.head(n=20)

,corpus,param,beta,std,k,se,t,p
0,CABNC,Intercept,-0.142720,1.435215,1997,0.032116,-4.443834,0.000005
1,CABNC,nx,0.175983,0.122897,1997,0.002750,63.990583,0.000000
2,CABNC,ny,-0.031646,0.051190,1997,0.001145,-27.626784,0.000000
3,CABNC,other,0.110075,1.523118,1997,0.034084,3.229567,0.000630
4,CABNC,self,-0.200672,2.052572,1997,0.045931,-4.368957,0.000007
5,CABNC,turn_delta_other,-0.018698,0.522819,1997,0.011699,-1.598241,0.055074
6,CABNC,turn_delta_self,0.007250,1.037192,1997,0.023210,0.312355,0.377402
7,CANDOR,Intercept,0.110503,0.379277,1653,0.009329,11.845542,0.000000
8,CANDOR,nx,0.135678,0.017057,1653,0.000420,323.410690,0.000000
9,CANDOR,ny,-0.023497,0.004826,1653,0.000119,-197.947452,0.000000


In [18]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'by-corpus-results.csv'),
    index=False, encoding='utf-8'
)

## Wald procedure

In [19]:
from shared.wald_test import wald

In [20]:
X = np.concat([df_regression['beta'].loc[df_regression['param'].isin([param])].values.reshape(-1,1) for param in param_list+processed_params], axis=-1)
X.shape

(4518, 7)

In [21]:
W = wald(X)
W.beta.shape

(7,)

In [22]:
print(param_list+processed_params)

['nx', 'ny', 'Intercept', 'self', 'other', 'turn_delta_self', 'turn_delta_other']


### Self versus other model

In [23]:
contrast = np.array([[0,0,0,1,-1,1,-1]])

contrast @ W.beta, W.test(contrast)

(array([-0.27989344]), (np.float64(338.50784006665685), 1, np.float64(0.0)))

### Self model

In [24]:
contrast = np.array([[0,0,0,1,0,1,0]])

contrast @ W.beta, W.test(contrast)

(array([-0.20332681]), (np.float64(275.4880966697742), 1, np.float64(0.0)))

### Other model

In [25]:
contrast = np.array([[0,0,0,0,1,0,1]])

contrast @ W.beta, W.test(contrast)

(array([0.07656663]),
 (np.float64(33.185811061681996), 1, np.float64(8.375949489547452e-09)))

## Other statistics

In [26]:
dfstats = df_regression[['corpus', 'cid']].drop_duplicates(subset=['corpus', 'cid']).groupby(by=['corpus']).agg('count').reset_index()

In [27]:
# kspeakers
s = df_regression[['corpus', 'cid', 'nspeakers']].drop_duplicates(subset=['corpus', 'cid'])
dfstats['speakers'] = s[['corpus', 'nspeakers']].groupby(by=['corpus']).agg('sum').reset_index()['nspeakers']
dfstats.head()

,corpus,cid,speakers
0,CABNC,1997,6126
1,CANDOR,1653,3306
2,CORAAL,271,588
3,DISPEL,19,38
4,Frederiksen,2,4


In [28]:
# comparisons
s = df_regression[['corpus', 'cid', 'k']].drop_duplicates(subset=['corpus', 'cid'])
dfstats['comparisons'] = s[['corpus', 'k']].groupby(by=['corpus']).agg('sum').reset_index()['k']
dfstats.head()

,corpus,cid,speakers,comparisons
0,CABNC,1997,6126,33778347
1,CANDOR,1653,3306,76594140
2,CORAAL,271,588,37093700
3,DISPEL,19,38,457869
4,Frederiksen,2,4,3956


In [29]:
dfstats.to_csv(
    os.path.join(OUTPUTS_PATH, 'corpus-descriptive-dfstats.csv'),
    index=False,
    encoding='utf-8'
)

In [30]:
dfstats['comparisons'].sum()

np.int64(203059333)

In [31]:
dfstats['cid'].sum()

np.int64(4518)